# Détection de fraude bancaire
Projet de machine learning supervisé, par Ahmed Amine El Amri.

On apprend à un modèle à reconnaître les transactions frauduleuses à partir d'un historique où on connaît déjà la réponse (`isFraud`). Données : PaySim, ~6,3 millions de transactions, 0,13 % de fraude.

### Préparation de l'environnement
On se place à la racine du projet pour que les chemins `data/...` fonctionnent, que le notebook soit lancé depuis la racine ou depuis `notebooks/`.

In [18]:
import os
if os.path.isdir('../data') and not os.path.isdir('data'):
    os.chdir('..')
print('Dossier de travail :', os.getcwd())

Dossier de travail : /home/ubuntu/Bureau/detection-fraude


## 1. Exploration des données (EDA)

On regarde les données pour les comprendre et repérer ce qui distingue une fraude. J'avais d'abord pris une version synthétique de PaySim, mais elle n'avait aucun signal (AUC de 0,49, du pur hasard). Je suis passé au vrai PaySim, ce que l'exploration permet d'attraper à temps.

In [19]:
import pandas as pd
df = pd.read_csv('data/paysim_real.csv')
print('Dimensions :', df.shape)
df.head(3)

Dimensions : (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0


### 1.1 Le déséquilibre

In [20]:
counts = df['isFraud'].value_counts().sort_index()
print('Normales :', counts[0])
print('Fraudes :', counts[1], f'({100*counts[1]/len(df):.3f} %)')

Normales : 6354407
Fraudes : 8213 (0.129 %)


Avec 0,13 % de fraude, l'accuracy ne veut rien dire. On regardera precision et recall.

### 1.2 La fraude selon le type de transaction

In [21]:
taux = (df.groupby('type')['isFraud'].mean()*100).sort_values(ascending=False)
print(taux.round(3))

type
TRANSFER    0.769
CASH_OUT    0.184
CASH_IN     0.000
DEBIT       0.000
PAYMENT     0.000
Name: isFraud, dtype: float64


La fraude n'apparaît que sur TRANSFER et CASH_OUT. Le fraudeur transfère l'argent volé puis le retire.

### 1.3 Le schéma du compte vidé

In [22]:
fr = df[df.isFraud==1]; no = df[df.isFraud==0]
print('Prend tout le solde, fraude : %.1f %%' % (100*(fr.amount==fr.oldbalanceOrg).mean()))
print('Prend tout le solde, normale : %.1f %%' % (100*(no.amount==no.oldbalanceOrg).mean()))

Prend tout le solde, fraude : 97.8 %
Prend tout le solde, normale : 0.0 %


Près de 98 % des fraudes prennent exactement tout le solde, contre quasiment jamais pour une normale.

## 2. Préparation des données

### 2.1 Garder les bonnes colonnes et les bonnes lignes

Je retire les identifiants et l'ancienne alarme inutile, et je me concentre sur TRANSFER et CASH_OUT.

In [23]:
df = df.drop(columns=['nameOrig','nameDest','isFlaggedFraud'])
df = df[df['type'].isin(['TRANSFER','CASH_OUT'])].copy()
print(df.shape, '- taux de fraude : %.3f %%' % (100*df.isFraud.mean()))

(2770409, 8) - taux de fraude : 0.296 %


### 2.2 Fabriquer des variables (feature engineering)

In [24]:
df['vide_tout'] = (df['amount'] == df['oldbalanceOrg']).astype(int)
df['erreur_emetteur'] = df['oldbalanceOrg'] - df['amount'] - df['newbalanceOrig']
df['erreur_dest']     = df['newbalanceDest'] - df['oldbalanceDest'] - df['amount']
df[['vide_tout','erreur_emetteur','erreur_dest']].head()

,vide_tout,erreur_emetteur,erreur_dest
2,1,0.00,-181.0
3,1,0.00,-21363.0
15,0,-213808.94,-182703.5
19,0,-214605.30,-237735.3
24,0,-300850.89,2401220.0


`vide_tout` vaut 1 pour presque toutes les fraudes. J'ajoute deux variables de cohérence des soldes pour que le modèle ne repose pas que sur elle.

### 2.3 Traduire le type en chiffre

In [25]:
df['est_transfer'] = (df['type'] == 'TRANSFER').astype(int)
df = df.drop(columns=['type'])
df.to_csv('data/prepare.csv', index=False)
print('Tableau prêt :', df.shape)
df.head(3)

Tableau prêt : (2770409, 11)


,step,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,vide_tout,erreur_emetteur,erreur_dest,est_transfer
2,1,181.00,181.0,0.0,0.0,0.00,1,1,0.00,-181.0,1
3,1,181.00,181.0,0.0,21182.0,0.00,1,1,0.00,-21363.0,0
15,1,229133.94,15325.0,0.0,5083.0,51513.44,0,0,-213808.94,-182703.5,0


## 3. Séparer entraînement et test

Une partie pour apprendre, une autre jamais vue pour juger. `stratify` garde le même taux de fraude des deux côtés.

In [26]:
from sklearn.model_selection import train_test_split
X = df.drop(columns=['isFraud']); y = df['isFraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
print('Entraînement :', X_train.shape[0], '-', int(y_train.sum()), 'fraudes')
print('Test :', X_test.shape[0], '-', int(y_test.sum()), 'fraudes')

Entraînement : 1939286 - 5749 fraudes
Test : 831123 - 2464 fraudes


## 4. Entraîner le modèle

Une régression logistique. Je mets les variables à la même échelle et `class_weight='balanced'` pour que le modèle fasse attention à la fraude malgré sa rareté.

In [27]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib

scaler = StandardScaler().fit(X_train)
modele = LogisticRegression(max_iter=1000, class_weight='balanced')
modele.fit(scaler.transform(X_train), y_train)

poids = pd.Series(modele.coef_[0], index=X.columns).sort_values()
print(poids.round(2))

joblib.dump(modele, 'modele_logistique.joblib')
joblib.dump(scaler, 'scaler.joblib')

oldbalanceDest     -2.54
newbalanceDest     -2.16
step               -1.99
erreur_emetteur    -0.34
est_transfer       -0.21
newbalanceOrig     -0.16
erreur_dest         0.31
oldbalanceOrg       0.44
amount              0.48
vide_tout          14.26
dtype: float64


['scaler.joblib']

Le modèle donne de loin le plus gros poids à `vide_tout` : il a retrouvé tout seul le signal repéré à l'exploration.

## 5. Évaluer le modèle

Test sur les données jamais vues. On regarde la matrice de confusion, le recall et la precision.

In [28]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
y_pred = modele.predict(scaler.transform(X_test))
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print('Fraudes attrapées :', tp, '| ratées :', fn, '| fausses alertes :', fp)
print('Recall    : %.1f %%' % (100*recall_score(y_test, y_pred)))
print('Precision : %.1f %%' % (100*precision_score(y_test, y_pred)))
print('F1-score  : %.1f %%' % (100*f1_score(y_test, y_pred)))

Fraudes attrapées : 2453 | ratées : 11 | fausses alertes : 22
Recall    : 99.6 %
Precision : 99.1 %
F1-score  : 99.3 %


Le modèle attrape 2 453 fraudes sur 2 464 (recall 99,6 %) avec 22 fausses alertes (precision 99,1 %). Ces scores élevés tiennent beaucoup à `vide_tout`, presque parfaite dans PaySim. Sur des fraudes réelles, plus subtiles, on n'atteindrait pas ce niveau.